In [1]:
import numpy as np
from scipy.optimize import curve_fit

# Example data: quark masses (in GeV) and geodesic lengths (approximate)
quark_masses = np.array([0.0022, 0.0047, 0.096, 1.28, 4.18, 173])
geodesic_lengths = np.array([0.82, 1.01, 1.99, 2.36, 3.12, 3.79])

# The model: m_q = m0 * exp(alpha * l^beta)
def mass_formula(l, m0, alpha, beta):
    return m0 * np.exp(alpha * l**beta)

# Fit parameters to the data in log-domain for stability
def log_mass_formula(l, log_m0, alpha, beta):
    return log_m0 + alpha * l**beta

log_masses = np.log(quark_masses)

# Fit using curve_fit
popt, pcov = curve_fit(log_mass_formula, geodesic_lengths, log_masses, p0=[-7, 1, 1])
log_m0, alpha, beta = popt
m0 = np.exp(log_m0)

print("Fitted parameters:")
print(f"m0 = {m0:.4e} GeV")
print(f"alpha = {alpha:.4f}")
print(f"beta = {beta:.4f}")

# Function to compute McShane arctan-sum constraint - returning sum value
def mcshane_sum(masses, m0, alpha, beta):
    n = len(masses)
    result = 0.0
    for i in range(n):
        for j in range(n):
            if i != j:
                # Inverse mapping to lengths via: l = [1/alpha * log(m_i/m0)]^{1/beta}
                li = (1/alpha * np.log(masses[i]/m0))**(1/beta)
                lj = (1/alpha * np.log(masses[j]/m0))**(1/beta)
                numerator = 2 * np.cosh(0.5 * li - 0.25 * lj)
                denominator = np.sinh(li) + np.sinh(0.5 * lj)
                term = np.arctan(numerator / denominator)
                result += term
    return result

# Calculate McShane sum and compare to π/2 (approximately 1.5708)
sum_value = mcshane_sum(quark_masses, m0, alpha, beta)
print(f"McShane sum: {sum_value:.4f} (expected pi/2 ≈ {np.pi/2:.4f})")

# Compare the predicted masses using fitted parameters
predicted_masses = mass_formula(geodesic_lengths, m0, alpha, beta)
for i, (real_m, pred_m) in enumerate(zip(quark_masses, predicted_masses)):
    print(f"Quark {i+1}: real mass = {real_m:.4e} GeV, predicted mass = {pred_m:.4e} GeV")

Fitted parameters:
m0 = 2.1306e-04 GeV
alpha = 3.0019
beta = 1.1167
McShane sum: 13.0974 (expected pi/2 ≈ 1.5708)
Quark 1: real mass = 2.2000e-03 GeV, predicted mass = 2.3609e-03 GeV
Quark 2: real mass = 4.7000e-03 GeV, predicted mass = 4.4339e-03 GeV
Quark 3: real mass = 9.6000e-02 GeV, predicted mass = 1.3797e-01 GeV
Quark 4: real mass = 1.2800e+00 GeV, predicted mass = 5.3638e-01 GeV
Quark 5: real mass = 4.1800e+00 GeV, predicted mass = 9.4091e+00 GeV
Quark 6: real mass = 1.7300e+02 GeV, predicted mass = 1.2606e+02 GeV
